In [ ]:
import os, sys
sys.path.insert(0, os.environ.get("GMS_RAG_TUTORIAL", os.path.dirname(os.getcwd())))
from book_kit import load_store, CORPUS, STORE, EVAL_COHORT, DEVICE, store_config

# Ch16 — Capstone: a complete GEODE-RAG over the annual report

This notebook assembles every part of the book into one pipeline over Northwind Industries FY2025: a GEODE-corrected store, triple-mediated retrieval with binding, multi-hop, provenance, self-verification, honest abstention, calibrated thresholds, and KAL persistence — then runs the full labeled cohort and reports the trust metrics a release would gate on.

**No GPU in this draft.** We load the *already-built* store from `data/gms_annual_report_store/` (built once by `scripts/build_store.py`) and use a deterministic scripted `LLMBackend` so the notebook runs in CI without Qwen. The real Qwen wiring is shown in one listing, marked for the lead to run.

## 1. Load the prebuilt, GEODE-corrected store
Ingestion + GEODE self-correction already happened in `build_store.py` (Ch4, Ch5). The capstone consumes the artifact; it does not rebuild it.

In [ ]:
store = load_store()
print("triples:", len(store.query_triples()))

## 2. The synthesis/extraction LLM
Two paths. In CI we inject a deterministic scripted backend so the run is reproducible; the real deployment uses local Qwen2.5-3B-Instruct (no API key). Per-role backends are wired through `RagConfig` (Ch9, Ch14).

In [ ]:
from knowlytix.knowledge.llm_backend import LLMBackend


class CapstoneLLM(LLMBackend):
    """Deterministic backend for CI. Extracts query triples for the cohort
    questions and synthesizes from the evidence block — no parametric guesses."""

    # NL question -> a single query triple with one "?" asked slot.
    _EXTRACT = {
        "cloud platform revenue": '[{"head":"cloud platform","relation":"has_revenue","tail":"?"}]',
        "total revenue": '[{"head":"total","relation":"has_revenue","tail":"?"}]',
        "retail": '[{"head":"retail","relation":"has_headcount","tail":"?"}]',
        "net income": '[{"head":"net income","relation":"has_fy2025","tail":"?"}]',
        "shareholders equity": '[{"head":"shareholders equity","relation":"has_amount","tail":"?"}]',
        "ceo": '[{"head":"ceo","relation":"has_value","tail":"?"}]',
        "topline": '[{"head":"total","relation":"has_revenue","tail":"?"}]',
        "cloud platform": '[{"head":"cloud platform","relation":"has_division","tail":"?x"},'
                          '{"head":"?x","relation":"has_region","tail":"?"}]',
        "logistics": '[{"head":"logistics","relation":"has_division","tail":"?x"},'
                     '{"head":"?x","relation":"has_head","tail":"?"}]',
    }

    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        q = user.lower()
        # Extraction call: return query triples keyed off the question.
        if "json" in system.lower() or "relation" in system.lower():
            for key, payload in self._EXTRACT.items():
                if key in q:
                    return payload
            return "[]"
        # Synthesis call: answer ONLY from the FACT lines in the evidence.
        facts = [ln for ln in user.splitlines() if ln.startswith("FACT:")]
        if not facts:
            return "I cannot answer this from the available grounded evidence."
        tail = facts[-1].split("|")[-1].strip()
        return f"The value is {tail}."

    @property
    def model_name(self) -> str:
        return "capstone-scripted"


ci_llm = CapstoneLLM()

**Real Qwen path (for the lead to run on the GPU box).** Same `RagConfig`, real backend — drop in `qwen_agent_callable` / a `LocalTransformersBackend`. This cell is illustrative and not executed in the CPU draft.

In [ ]:
# RUN-ON-GPU (lead): swap the scripted backend for local Qwen2.5-3B-Instruct.
# from knowlytix.knowledge.llm_backend import LocalTransformersBackend
# from knowlytix.knowledge.geode import QWEN_3B
# qwen = LocalTransformersBackend(QWEN_3B, device=device)
# rag_real = RagConfig(llm=qwen)            # see assembly below
pass

## 3. Assemble the full bank-grade pipeline
Triple-mediated by default: dense is off, the relevance gate is on, answers self-verify, and the pipeline abstains rather than guess. We set `on_verify_fail="abstain"` so a hallucinated number cannot survive (Ch10).

In [ ]:
from knowlytix.knowledge.rag import RagConfig, RagPipeline

rag = RagConfig(
    llm=ci_llm,                 # synthesis
    llm_extract=ci_llm,         # NL -> query triples
    binding="fuzzy",            # Ch7; embedding mode shown there
    ground_extraction=True,     # inject the graph schema (Ch6)
    relevance_gate=False,       # scripted LLM has no relevance judge; off in CI
    verify_llm_output=True,     # self-verify the draft (Ch10)
    on_verify_fail="abstain",   # a contradicted claim -> abstain, never emit
    dense_fallback=False,       # distrusted dense stays OFF (the thesis)
    accept_threshold=0.0,       # calibrated below
)
pipe = RagPipeline.from_store(store, rag)
print("pipeline ready; dense_fallback =", rag.dense_fallback)

## 4. A grounded answer with provenance
Single-hop: Cloud Platform revenue. The answer carries the ENM-exact figure and a `file:line:char` span (Ch3, Ch8).

In [ ]:
ans = pipe.query("What is Cloud Platform revenue?")
print("decision:", ans.decision, "| verified:", ans.verified)
print("answer  :", ans.answer)
for f in ans.sources:
    print("  fact:", (f.head, f.relation, f.tail), "@", f.location)
assert ans.decision == "accept"
assert any(f.tail == "120.0" for f in ans.sources)

## 5. Multi-hop through the hierarchy
`?x -> ?`: which region runs the division that contains Cloud Platform? The retriever resolves `has_division` then `has_region` (Ch8). Both hops appear as facts in the audit.

In [ ]:
mh = pipe.query("Which region runs the division that contains Cloud Platform?")
print("decision:", mh.decision, "| answer:", mh.answer)
for f in mh.sources:
    print("  hop:", (f.head, f.relation, f.tail))
assert mh.decision == "accept"
assert any(f.tail == "north america" for f in mh.sources)

## 6. Honest abstention on a prose blind spot
The Outlook section carries no triples — it is a measurable coverage blind spot (Ch11). The pipeline abstains instead of guessing from prose, and `coverage_report` names the blind spots.

In [ ]:
from knowlytix.knowledge.rag import coverage_report

out = pipe.query("What is management's outlook for fiscal 2026?")
print("decision:", out.decision, "| notice:", out.notice)
cov = coverage_report(store)
blind = [r.title for r in cov.blind_spots]
print("coverage_ratio:", round(cov.coverage_ratio, 2))
print("blind spots   :", blind)
assert out.decision == "abstain"
assert any("Outlook" in b for b in blind)

## 7. Self-verification catches a confident-wrong number
We force the synthesis LLM to emit a *wrong* Cloud Platform figure. The GMS self-verifier (Ch10) decomposes the draft into claim triples, finds the tail contradicts the asserted fact (`120.0`), and the pipeline abstains — no confident-wrong answer escapes.

In [ ]:
class HallucinatingLLM(CapstoneLLM):
    """Same extraction; synthesis lies about the Cloud Platform figure."""
    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        if not ("json" in system.lower() or "relation" in system.lower()):
            return "Cloud Platform revenue was 999.0."  # hallucinated tail
        return super().call(system, user, max_tokens)

bad_rag = RagConfig(llm=HallucinatingLLM(), llm_extract=ci_llm,
                    relevance_gate=False, verify_llm_output=True,
                    on_verify_fail="abstain")
bad_pipe = RagPipeline.from_store(store, bad_rag)
bad = bad_pipe.query("What is Cloud Platform revenue?")
print("decision:", bad.decision, "| verification ok:", bad.verification.get("ok"))
assert bad.verification.get("ok") is False
assert bad.decision == "abstain"

## 8. Calibrate, don't guess
Thresholds come from the labeled cohort, not from eyeballing (Ch12). Here we load `eval_cohort.json` and keep the bank-grade `accept_threshold=0.0` (every grounded answer is accepted; abstention is driven by binding + verification, not a confidence cut on this corpus). The cohort is the calibration set.

In [ ]:
import json

with open(EVAL_COHORT) as fh:
    cohort = json.load(fh)
print("cohort size:", len(cohort), "| types:",
      sorted({c["type"] for c in cohort}))

## 9. Run the full cohort and report trust metrics
`evaluate` runs every labeled question through the pipeline and scores bind-rate, answer accuracy, decision accuracy (accept/abstain correctness) (Ch13). The release gate is **zero confident-wrong**: no `accept` on a case whose answer is wrong, and no `accept` on an unanswerable case.

In [ ]:
from knowlytix.knowledge.rag import EvalCase, evaluate

cases = [EvalCase(question=c["question"],
                  expected_answer=c.get("expected_answer"),
                  expect_decision=c.get("expect_decision"))
         for c in cohort]
report = evaluate(pipe, cases)
m = report.as_dict()
print("n              :", m["n"])
print("bind_rate      :", round(m["bind_rate"], 3))
print("answer_accuracy:", round(m["answer_accuracy"], 3))
print("decision_accy  :", round(m["decision_accuracy"], 3))

# Trust gate: zero confident-wrong. A case is confident-wrong if it was
# accepted but the decision (vs its label) was wrong.
confident_wrong = [r for r in report.results
                   if r.decision == "accept" and r.decision_correct is False]
print("confident_wrong:", len(confident_wrong))

## 10. Provenance rate
A trustworthy answer is a *cited* answer. We measure the share of accepted answers that carry at least one source span (Ch3).

In [ ]:
def provenance_rate(pipe, cohort):
    accepted, with_span = 0, 0
    for c in cohort:
        a = pipe.query(c["question"])
        if a.decision != "accept":
            continue
        accepted += 1
        if any(f.location for f in a.sources):
            with_span += 1
    return with_span / accepted if accepted else 0.0

prov = provenance_rate(pipe, cohort)
print("provenance_rate:", round(prov, 3))

## 11. Persist the graph to KAL (offline mock)
The corrected, verified graph lives in a real KG store. We convert the store's triples to `KALTriple` records (provenance + verification carried) and persist through KAL's offline mock adapter, then confirm the round-trip count (Ch15). The Postgres path is the same call against `kal_postgres_adapter(...)`, gated on a DSN env var.

In [ ]:
from knowlytix.kal.adapters import MockKnowledgeAdapter
from knowlytix.knowledge.rag.kal_sink import (
    persist_store_to_kal, store_to_kal_triples,
)

kal_triples = store_to_kal_triples(store, source="annual_report.md",
                                   confidence=1.0)
adapter = MockKnowledgeAdapter("capstone")
# Notebooks already run inside an event loop, so await the coroutine directly
# (asyncio.run() would raise "cannot be called from a running event loop").
n = await persist_store_to_kal(adapter, store, tenant_id="northwind",
                               source="annual_report.md", confidence=1.0)
print("persisted:", n, "of", len(kal_triples), "triples")
assert n == len(kal_triples)

## 12. The bridge: wrap the pipeline as a `search_policy` tool
The capstone is also the hand-off to *Beyond Prompt and Pray*. We wrap `pipe.query` as a typed, gated tool that returns a grounded answer **with** its provenance and audit trail — the shape a governed agent's executor expects. The full governed-loop wiring is Appendix~\ref{app:agent-bridge}; here we show the tool surface.

In [ ]:
def search_policy(question: str) -> dict:
    """Grounded RAG tool for a governed agent: answer + provenance + audit."""
    a = pipe.query(question)
    return {
        "answer": a.answer,
        "decision": a.decision,        # accept | abstain | escalate
        "verified": a.verified,        # GMS-verified, not dense
        "provenance": [f.location for f in a.sources if f.location],
        "audit": a.audit(),            # full structured trail
    }

tool_out = search_policy("What was total revenue?")
print("answer    :", tool_out["answer"])
print("decision  :", tool_out["decision"])
print("provenance:", tool_out["provenance"])

## 13. Self-check — the capstone claim
The full cohort run hits its trust targets *and* the audit trail is complete: **zero confident-wrong**, every accepted answer cited, every unanswerable case abstained, and the persisted-graph round-trip is exact.

In [ ]:
# Trust gate (the chapter's claim).
assert len(confident_wrong) == 0, confident_wrong

# Every unanswerable/prose case abstains.
for c in cohort:
    if c.get("expect_decision") == "abstain":
        assert pipe.query(c["question"]).decision == "abstain", c["id"]

# Every accepted answer is cited (provenance rate == 1.0 on this corpus).
assert prov == 1.0, prov

# Audit trail is complete for an accepted answer.
audit = pipe.query("What is Cloud Platform revenue?").audit()
assert audit["decision"] == "accept"
assert audit["sources"] and audit["sources"][0]["location"]
assert audit["route"] == "triple" and audit["verified"] is True

# Graph fully persisted to KAL.
assert n == len(kal_triples)

print("CAPSTONE OK: zero confident-wrong, full provenance, complete audit.")